In [6]:
import re
from datetime import datetime
import pandas as pd
from dateutil import parser as dateparser

LINE_RE = re.compile(r"^\[(.+?)\] ([^:]+): (.*)$")

def parse_export(path):
       messages = []
       current = None
       for line in open(path, encoding="utf-8"):
           m = LINE_RE.match(line)
           if m:
               if current:
                   messages.append(current)
               ts, sender, body = m.groups()
               current = {
                   "timestamp": dateparser.parse(ts, fuzzy=True),
                   "sender": sender.strip(),
                   "body": body.rstrip("\n"),
               }
           elif current:
               current["body"] += "\n" + line.rstrip("\n")
       if current:
           messages.append(current)
       df = pd.DataFrame(messages)
       df["is_media"] = df["body"].str.contains("<Media omitted>", na=False)
       df["is_system"] = df["body"].str.contains("added|removed|changed|created", case=False, na=False)
       return df

In [8]:
df = parse_export("sample_chat.txt")
print(df.head())

            timestamp      sender                                    body  \
0 2026-03-15 14:23:01    John Doe                   Welcome to the group!   
1 2026-03-15 14:24:15  Jane Smith                         <Media omitted>   
2 2026-03-15 14:25:30    John Doe  When is the next meeting we agreed on?   
3 2026-03-15 14:26:02      System               John Doe added Bob Wilson   
4 2026-03-15 14:27:10  Bob Wilson       Hi everyone, thanks for adding me   

   is_media  is_system  
0     False      False  
1      True      False  
2     False      False  
3     False       True  
4     False      False  


First we cover the easy anaytics:
1. Hour frequency
2. User frequency
3. Daily and weekly frequencies

In [29]:
# frequency by hour (peak hours)
print (df.groupby(df["timestamp"].dt.hour).size().sort_values(ascending=False))

# frequency by sender (per user)
print (df.groupby(df["sender"]).size().sort_values(ascending=False))

# frequency by day and week (per date)
print (df.groupby(df["timestamp"].dt.date).size())
print (df.groupby(df["timestamp"].dt.isocalendar().week).size())

# summary (System sender included)
print (len(df)) #total message count
print (len(df[~df['is_media'] & ~df['is_system']])) #non media and non system message count
print (len(df.groupby(df["sender"]))) #unique sender count
print (df['is_media'].sum()) #media message count
print (df["timestamp"].min()) #first message date
print (df["timestamp"].max()) #last message date


timestamp
14    38
9     32
15    26
dtype: int64
sender
Jane Smith      24
John Doe        19
Mike Brown      17
Bob Wilson      15
Sarah Connor    12
System           9
dtype: int64
timestamp
2026-03-15    64
2026-03-16    32
dtype: int64
week
11    64
12    32
dtype: int64
96
89
6
1
2026-03-15 14:23:01
2026-03-16 09:31:40
